# NB1 — Clean Fashion-MNIST baseline


In [1]:
QUICK = True

import copy
import hashlib
import json
import os
import random
import statistics
import sys
import time
from pathlib import Path
from uuid import uuid4

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from IPython.display import Markdown, display

START_TIME = time.perf_counter()
PROJECT_ROOT = Path.cwd()
CACHE_ROOT = PROJECT_ROOT / "results" / "nb1"

SPLIT_SEED = 42
TRAIN_SIZE = 50000
VAL_SIZE = 10000
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATES = [0.0003, 0.001, 0.003]
TUNING_SEED = 123
TRAIN_SEEDS = [0, 1, 2]

# Use CPU on both local and course systems. All loaders use num_workers=0.
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.use_deterministic_algorithms(True)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Mode:", "QUICK" if QUICK else "FULL")

Python: 3.13.11
PyTorch: 2.10.0+cu130
Mode: QUICK


## Model, randomness and cache


In [2]:
def build_model():
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(32 * 7 * 7, 10),
    )


def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)


def make_loader(dataset, seed, shuffle):
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=torch.Generator().manual_seed(seed),
        num_workers=0,
    )


def save_torch_atomic(value, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid4().hex + ".tmp")
    try:
        torch.save(value, temporary)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


def save_json_atomic(value, path):
    temporary = path.with_name(path.name + "." + uuid4().hex + ".tmp")
    try:
        temporary.write_text(
            json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


CONFIG = {
    "experiment_version": 1,
    "dataset": "Fashion-MNIST clean, float32 pixels in [0, 1]",
    "model": str(build_model()),
    "optimizer": "Adam, other settings use PyTorch defaults",
    "split_seed": SPLIT_SEED,
    "train_size": TRAIN_SIZE,
    "val_size": VAL_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rates": LEARNING_RATES,
    "tuning_seed": TUNING_SEED,
    "training_seeds": TRAIN_SEEDS,
    "selection": "max validation accuracy, then min validation loss",
}
config_text = json.dumps(CONFIG, sort_keys=True)
config_id = hashlib.sha256(config_text.encode()).hexdigest()[:12]
CACHE_DIR = CACHE_ROOT / config_id
SUMMARY_PATH = CACHE_DIR / "summary.json"
print("Cache directory:", CACHE_DIR)
print("Model parameters:", sum(p.numel() for p in build_model().parameters()))

Cache directory: /uolstore/home/student_lnxhome01/fjts2709/Downloads/cw3test/results/nb1/51d47e24908a
Model parameters: 20490


## Data


In [3]:
if QUICK:
    if not SUMMARY_PATH.is_file():
        raise FileNotFoundError(
            "errors"
        )
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    if summary["config"] != CONFIG:
        raise ValueError("errors")
    print("Loaded full-experiment results. No dataset download is needed.")
else:
    if not (PROJECT_ROOT / "fashion_data.py").is_file():
        raise FileNotFoundError("errors")
    from fashion_data import load_minimum_data

    clean_train, patched_train, clean_test = load_minimum_data(
        PROJECT_ROOT / "data", download=True
    )
    train_images = clean_train.data.unsqueeze(1).to(torch.float32).div(255)
    test_images = clean_test.data.unsqueeze(1).to(torch.float32).div(255)
    assert torch.equal(train_images[0], clean_train[0][0])

    all_train_data = TensorDataset(train_images, clean_train.targets)
    test_data = TensorDataset(test_images, clean_test.targets)
    train_data, val_data = random_split(
        all_train_data,
        [TRAIN_SIZE, VAL_SIZE],
        generator=torch.Generator().manual_seed(SPLIT_SEED),
    )
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    save_torch_atomic({
        "split_seed": SPLIT_SEED,
        "train_indices": torch.tensor(train_data.indices),
        "val_indices": torch.tensor(val_data.indices),
    }, CACHE_DIR / "split_indices.pt")

    # These are real clean training/validation images, saved compactly for QUICK.
    demo_train_indices = train_data.indices[:512]
    demo_val_indices = val_data.indices[:256]
    save_torch_atomic({
        "config": CONFIG,
        "train_images": clean_train.data[demo_train_indices].clone(),
        "train_labels": clean_train.targets[demo_train_indices].clone(),
        "val_images": clean_train.data[demo_val_indices].clone(),
        "val_labels": clean_train.targets[demo_val_indices].clone(),
    }, CACHE_DIR / "quick_data.pt")
    print("Train / validation / test:", len(train_data), len(val_data), len(test_data))

Loaded full-experiment results. No dataset download is needed.


## Training and evaluation



In [4]:
criterion = nn.CrossEntropyLoss()


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    with torch.no_grad():
        for images, labels in loader:
            scores = model(images)
            loss = criterion(scores, labels)
            total_loss += loss.item() * labels.size(0)
            total_correct += (scores.argmax(dim=1) == labels).sum().item()
            total_examples += labels.size(0)
    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


def train_run(training_data, validation_data, seed, learning_rate, epochs):
    seed_everything(seed)
    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    training_loader = make_loader(training_data, seed, shuffle=True)
    validation_loader = make_loader(validation_data, seed, shuffle=False)
    history = []
    best_key = (-1.0, float("-inf"))
    best_state = None
    best_epoch = None
    started = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_examples = 0
        for images, labels in training_loader:
            optimizer.zero_grad()
            scores = model(images)
            loss = criterion(scores, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            total_correct += (scores.argmax(dim=1) == labels).sum().item()
            total_examples += labels.size(0)

        validation = evaluate(model, validation_loader)
        history.append({
            "epoch": epoch,
            "train_loss": total_loss / total_examples,
            "train_accuracy": total_correct / total_examples,
            "val_loss": validation["loss"],
            "val_accuracy": validation["accuracy"],
        })
        selection_key = (validation["accuracy"], -validation["loss"])
        if selection_key > best_key:
            best_key = selection_key
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
        print(
            f"seed={seed}, lr={learning_rate:g}, epoch={epoch}/{epochs}, "
            f"train loss={history[-1]['train_loss']:.4f}, "
            f"val loss={validation['loss']:.4f}, val acc={validation['accuracy']:.2%}",
            flush=True,
        )

    return {
        "seed": seed,
        "learning_rate": learning_rate,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_key[0],
        "best_val_loss": -best_key[1],
        "model_state": best_state,
        "history": history,
        "training_seconds": time.perf_counter() - started,
    }


def get_or_train_run(phase, seed, learning_rate):
    filename = f"{phase}_seed_{seed}_lr_{learning_rate:g}.pt"
    path = CACHE_DIR / filename
    signature = {"config": CONFIG, "phase": phase, "seed": seed, "lr": learning_rate}
    if path.is_file():
        cached = torch.load(path, map_location="cpu", weights_only=True)
        if cached.get("signature") != signature:
            raise ValueError(f"errors：{path}")
        print("Reusing:", filename, flush=True)
        return cached

    record = train_run(train_data, val_data, seed, learning_rate, EPOCHS)
    record["signature"] = signature
    record["checkpoint"] = filename
    save_torch_atomic(record, path)
    return record

## Learning-rate search and three-seed baseline



In [5]:
if not QUICK:
    tuning_records = []
    for learning_rate in LEARNING_RATES:
        print(f"\nLearning-rate search: {learning_rate:g}", flush=True)
        tuning_records.append(get_or_train_run("tuning", TUNING_SEED, learning_rate))

    selected = max(
        tuning_records,
        key=lambda record: (record["best_val_accuracy"], -record["best_val_loss"]),
    )
    selected_lr = selected["learning_rate"]
    print(f"\nSelected learning rate: {selected_lr:g}", flush=True)

    baseline_records = []
    for seed in TRAIN_SEEDS:
        print(f"\nFull baseline: seed {seed}", flush=True)
        baseline_records.append(get_or_train_run("baseline", seed, selected_lr))

    # The test set is first evaluated after tuning and all baseline runs finish.
    test_loader = make_loader(test_data, SPLIT_SEED, shuffle=False)
    baseline_rows = []
    for record in baseline_records:
        trained_model = build_model()
        trained_model.load_state_dict(record["model_state"])
        test_metrics = evaluate(trained_model, test_loader)
        baseline_rows.append({
            "seed": record["seed"],
            "best_epoch": record["best_epoch"],
            "val_accuracy": record["best_val_accuracy"],
            "test_accuracy": test_metrics["accuracy"],
            "test_loss": test_metrics["loss"],
            "training_seconds": record["training_seconds"],
            "checkpoint": record["checkpoint"],
            "history": record["history"],
        })

    summary = {
        "config": CONFIG,
        "python_version": sys.version.split()[0],
        "torch_version": str(torch.__version__),
        "cpu_threads": torch.get_num_threads(),
        "selected_learning_rate": selected_lr,
        "tuning": [{
            "learning_rate": record["learning_rate"],
            "best_epoch": record["best_epoch"],
            "val_accuracy": record["best_val_accuracy"],
            "val_loss": record["best_val_loss"],
        } for record in tuning_records],
        "baseline": baseline_rows,
    }
    save_json_atomic(summary, SUMMARY_PATH)
    print("\nFull results saved:", SUMMARY_PATH)

## Results and seed variability



In [6]:
tuning_table = [
    "| Learning rate | Best epoch | Validation accuracy | Validation loss |",
    "|---|---|---|---|",
]
for row in summary["tuning"]:
    tuning_table.append(
        f"| {row['learning_rate']:g} | {row['best_epoch']} | "
        f"{row['val_accuracy']:.2%} | {row['val_loss']:.4f} |"
    )
display(Markdown("\n".join(tuning_table)))
print("Selected learning rate:", summary["selected_learning_rate"])

baseline_table = [
    "| Seed | Best epoch | Validation accuracy | Test accuracy | Test loss |",
    "|---|---|---|---|---|",
]
for row in summary["baseline"]:
    baseline_table.append(
        f"| {row['seed']} | {row['best_epoch']} | {row['val_accuracy']:.2%} | "
        f"{row['test_accuracy']:.2%} | {row['test_loss']:.4f} |"
    )
display(Markdown("\n".join(baseline_table)))

test_accuracies = [row["test_accuracy"] for row in summary["baseline"]]
mean_accuracy = statistics.mean(test_accuracies)
std_accuracy = statistics.stdev(test_accuracies)
spread = max(test_accuracies) - min(test_accuracies)
print(f"Mean test accuracy: {mean_accuracy:.2%}")
print(f"Sample standard deviation: {100 * std_accuracy:.3f} percentage points")
print(f"Range: {min(test_accuracies):.2%} to {max(test_accuracies):.2%}")
print(f"Max-minus-min spread: {100 * spread:.3f} percentage points")

for run in summary["baseline"]:
    history_table = [
        f"### Seed {run['seed']}",
        "| Epoch | Train loss | Validation loss | Train accuracy | Validation accuracy |",
        "|---|---|---|---|---|",
    ]
    for row in run["history"]:
        history_table.append(
            f"| {row['epoch']} | {row['train_loss']:.4f} | {row['val_loss']:.4f} | "
            f"{row['train_accuracy']:.2%} | {row['val_accuracy']:.2%} |"
        )
    display(Markdown("\n\n".join(history_table[:1]) + "\n\n" + "\n".join(history_table[1:])))

| Learning rate | Best epoch | Validation accuracy | Validation loss |
|---|---|---|---|
| 0.0003 | 5 | 87.58% | 0.3488 |
| 0.001 | 5 | 89.84% | 0.2917 |
| 0.003 | 5 | 91.05% | 0.2586 |

Selected learning rate: 0.003


| Seed | Best epoch | Validation accuracy | Test accuracy | Test loss |
|---|---|---|---|---|
| 0 | 4 | 90.47% | 90.20% | 0.2860 |
| 1 | 5 | 90.85% | 89.89% | 0.2893 |
| 2 | 4 | 90.87% | 90.00% | 0.2782 |

Mean test accuracy: 90.03%
Sample standard deviation: 0.157 percentage points
Range: 89.89% to 90.20%
Max-minus-min spread: 0.310 percentage points


### Seed 0

| Epoch | Train loss | Validation loss | Train accuracy | Validation accuracy |
|---|---|---|---|---|
| 1 | 0.4607 | 0.3627 | 83.50% | 86.83% |
| 2 | 0.3080 | 0.2980 | 89.05% | 89.41% |
| 3 | 0.2694 | 0.2886 | 90.43% | 89.66% |
| 4 | 0.2474 | 0.2715 | 90.99% | 90.47% |
| 5 | 0.2283 | 0.2754 | 91.81% | 90.29% |

### Seed 1

| Epoch | Train loss | Validation loss | Train accuracy | Validation accuracy |
|---|---|---|---|---|
| 1 | 0.4700 | 0.3383 | 83.09% | 88.28% |
| 2 | 0.3121 | 0.3205 | 88.91% | 88.05% |
| 3 | 0.2781 | 0.2854 | 89.91% | 90.14% |
| 4 | 0.2526 | 0.2731 | 90.91% | 90.26% |
| 5 | 0.2387 | 0.2631 | 91.37% | 90.85% |

### Seed 2

| Epoch | Train loss | Validation loss | Train accuracy | Validation accuracy |
|---|---|---|---|---|
| 1 | 0.4469 | 0.3412 | 84.00% | 88.05% |
| 2 | 0.3035 | 0.3062 | 89.08% | 89.13% |
| 3 | 0.2648 | 0.2827 | 90.46% | 89.89% |
| 4 | 0.2406 | 0.2619 | 91.30% | 90.87% |
| 5 | 0.2242 | 0.2719 | 91.84% | 90.21% |

## QUICK training example and checkpoint check



In [7]:
if QUICK:
    quick_data = torch.load(
        CACHE_DIR / "quick_data.pt", map_location="cpu", weights_only=True
    )
    if quick_data["config"] != CONFIG:
        raise ValueError("QUICK errors。")
    demo_train = TensorDataset(
        quick_data["train_images"].unsqueeze(1).to(torch.float32).div(255),
        quick_data["train_labels"],
    )
    demo_val = TensorDataset(
        quick_data["val_images"].unsqueeze(1).to(torch.float32).div(255),
        quick_data["val_labels"],
    )
    demo = train_run(
        demo_train, demo_val, TRAIN_SEEDS[0],
        summary["selected_learning_rate"], epochs=1,
    )
    print(f"Small-example validation accuracy: {demo['best_val_accuracy']:.2%}")

    saved_run = torch.load(
        CACHE_DIR / summary["baseline"][0]["checkpoint"],
        map_location="cpu", weights_only=True,
    )
    if saved_run["signature"]["config"] != CONFIG:
        raise ValueError(" checkpoint errors")
    restored_model = build_model()
    restored_model.load_state_dict(saved_run["model_state"])
    restored_metrics = evaluate(
        restored_model, make_loader(demo_val, SPLIT_SEED, shuffle=False)
    )
    print(f"Restored model, small validation subset: {restored_metrics['accuracy']:.2%}")

print(f"Notebook execution time: {time.perf_counter() - START_TIME:.1f} seconds")
print("Files to keep with the notebook:", CACHE_DIR)
if not QUICK:
    print("next:QUICK = True，restart kernel 后 Run All。")

seed=0, lr=0.003, epoch=1/1, train loss=2.1067, val loss=1.6732, val acc=42.97%
Small-example validation accuracy: 42.97%
Restored model, small validation subset: 93.36%
Notebook execution time: 2.0 seconds
Files to keep with the notebook: /uolstore/home/student_lnxhome01/fjts2709/Downloads/cw3test/results/nb1/51d47e24908a
